# Uebersicht Regelungstechnik (Paketfunktionen)

Dieses Notebook fasst die wichtigsten Funktionen aus dem Paket zusammen.

- Einfach gehaltene Beispiele
- Klare Kennzeichnung von Eingabe und Ausgabe
- Plots werden direkt im Notebook angezeigt

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import sympy as sp

from regelungstechnik import (
    laplace_transform, inverse_laplace, partialbruchzerlegung,
    hurwitz_kriterium, routh_kriterium, nyquist_kriterium,
    reihenschaltung, parallelschaltung, rueckkopplung,
    sprungantwort_mit_fex, sprungfaehigkeit_realisierbarkeit,
    stationaere_abweichung, maximale_ueberschwingweite, ausregelzeit,
    reglerparameter_nach_verfahren, phasenkorrekturglied_auslegung, wurzelortsauslegung,
    zustandsraum_zu_uebertragungsfunktion, regelungsnormalform, transitionsmatrix,
    transitionsmatrix_symbolisch, jordan_normalform, poincare_klassifikation,
    plot_sprungantwort, plot_impulsantwort, plot_bode, plot_ortskurve,
    plot_nyquist, plot_pol_nullstellen, plot_wurzelortskurve, plot_poincare
)

sp.init_printing()
print("Imports erfolgreich.")

## 1. Laplace und Partialbruch

In [ ]:
t = sp.symbols('t', positive=True)
f_t = t * sp.exp(-2 * t)

print("EINGABE f(t):", f_t)
res_L = laplace_transform(f_t)
print("AUSGABE F(s):", res_L['ergebnis'])

res_iL = inverse_laplace(res_L['ergebnis'])
print("AUSGABE f(t) aus Ruecktransformation:", res_iL['ergebnis'])

res_pb = partialbruchzerlegung([1, 3], [1, 4, 3])
print("AUSGABE Partialbruch:", res_pb['ergebnis'])

## 2. Stabilitaet (Hurwitz, Routh, Nyquist)

In [ ]:
num = [1]
den = [1, 3, 2]
print("EINGABE num:", num)
print("EINGABE den:", den)

res_h = hurwitz_kriterium(den)
res_r = routh_kriterium(den)
res_ny = nyquist_kriterium(num, den)

print("AUSGABE Hurwitz stabil:", res_h['ergebnis']['stabil'])
print("AUSGABE Routh stabil:", res_r['ergebnis']['stabil'])
print("AUSGABE Nyquist stabil:", res_ny['ergebnis']['stabil'])
print("AUSGABE Nyquist Kennzahlen:", {k: v for k, v in res_ny['ergebnis'].items() if k not in ['w', 'nyquist']})

## 3. Blockschaltung und Reglerentwurf

In [ ]:
G1 = ([1], [1, 1])
G2 = ([2], [1, 2])
H = ([1], [1, 5])

print("EINGABE G1:", G1)
print("EINGABE G2:", G2)
print("EINGABE H:", H)

res_reihe = reihenschaltung(G1, G2)
res_parallel = parallelschaltung(G1, G2)
res_rk = rueckkopplung(res_reihe['ergebnis'], H, negativ=True)

print("AUSGABE Reihenschaltung:", res_reihe['ergebnis'])
print("AUSGABE Parallelschaltung:", res_parallel['ergebnis'])
print("AUSGABE Rueckkopplung:", res_rk['ergebnis'])

res_reg = reglerparameter_nach_verfahren('PI', 'ziegler-nichols', modus='offen', K=2.0, T=5.0, K_T=1.0)
res_pk = phasenkorrekturglied_auslegung('anhebend', phi_grad=35.0, omega_c=2.0, K=1.0)
res_wurzel = wurzelortsauslegung([1], [1, 3, 2], k_start=0.0, k_ende=8.0, anzahl_k=41)

print("AUSGABE Reglerparameter:", res_reg['ergebnis'])
print("AUSGABE Phasenkorrekturglied:", res_pk['ergebnis'])
print("AUSGABE k_empfohlen:", res_wurzel['ergebnis']['k_empfohlen'])

res_sf = sprungfaehigkeit_realisierbarkeit([1], [1, 3, 2])
res_sa = stationaere_abweichung([1], [1, 3, 2])
res_mu = maximale_ueberschwingweite([1], [1, 2, 1])
res_az = ausregelzeit([1], [1, 2, 1], toleranzband=0.05)
res_fex = sprungantwort_mit_fex(([1], [1, 3, 2]), F_ex=([1], [1, 1]), t_ende=8.0)

print("AUSGABE Sprungfaehigkeit:", res_sf['ergebnis'])
print("AUSGABE Stationaere Abweichung:", res_sa['ergebnis'])
print("AUSGABE Ueberschwingweite:", res_mu['ergebnis'])
print("AUSGABE Ausregelzeit:", res_az['ergebnis'])
print("AUSGABE Sprungantwort mit F_ex (Signalpunkte):", len(res_fex['ergebnis'][0]))

## 4. Zustandsraum und Jordan

In [ ]:
A = [[0, 1], [-2, -3]]
B = [[0], [1]]
C = [[1, 0]]
D = [[0]]

print("EINGABE A,B,C,D:")
print(A)
print(B)
print(C)
print(D)

res_tf = zustandsraum_zu_uebertragungsfunktion(A, B, C, D)
num_tf, den_tf = res_tf['ergebnis']
print("AUSGABE Uebertragungsfunktion:", (num_tf, den_tf))

res_rnf = regelungsnormalform(num_tf, den_tf)
print("AUSGABE Regelungsnormalform A:")
print(res_rnf['ergebnis'][0])

res_tm = transitionsmatrix(A, [0.0, 0.5, 1.0])
print("AUSGABE Transition numerisch t=0.5:")
print(res_tm['ergebnis'][1])

res_tms = transitionsmatrix_symbolisch(A)
print("AUSGABE Transition symbolisch:")
display(res_tms['ergebnis'])

res_j = jordan_normalform(A)
res_p = poincare_klassifikation(A)
print("AUSGABE Jordanform J:")
display(res_j['ergebnis']['J'])
print("AUSGABE Poincare-Klassifikation:", res_p['ergebnis'])

## 5. Plotfunktionen (direkte Anzeige)

In [ ]:
num = [1]
den = [1, 3, 2]

plot_results = [
    plot_sprungantwort(num, den, dateiname='overview_sprung.png'),
    plot_impulsantwort(num, den, dateiname='overview_impuls.png'),
    plot_bode(num, den, dateiname='overview_bode.png'),
    plot_ortskurve(num, den, dateiname='overview_ortskurve.png'),
    plot_nyquist(num, den, dateiname='overview_nyquist.png'),
    plot_pol_nullstellen(num, den, dateiname='overview_polnull.png'),
    plot_wurzelortskurve(num, den, dateiname='overview_wurzelort.png'),
]
poincare_path = plot_poincare([[0, 1], [-2, -3]], dateiname='overview_poincare.png')

print("AUSGABE Plotdateien:")
for item in plot_results:
    pfad = item.get('plot_pfad')
    print(pfad)
    if pfad and Path(pfad).exists():
        display(Image(filename=pfad))

print(poincare_path)
if Path(poincare_path).exists():
    display(Image(filename=poincare_path))